# 🔐 ALL-IN-One IPTV — Playlist Encryptor

Encrypt your private M3U playlists with **AES-256-GCM** encryption and upload them to the GitHub `input/` folder.

The engine will auto-decrypt and merge your private streams with the public playlists.

---

## How It Works
1. Generate or set your encryption key (32 bytes / 64 hex chars)
2. Upload your `.m3u` playlist file
3. Run the encryption cell
4. Download the `.enc` file
5. Upload to `input/` folder on GitHub
6. Set `ENCRYPT_KEY` as a GitHub Secret

> ⚠️ **Save your key!** Without it, you cannot decrypt your playlist.

In [ ]:
# Install required library
!pip install pycryptodome -q

In [ ]:
import os
from google.colab import files

# Upload your M3U file
print("Upload your .m3u or .m3u8 playlist file:")
uploaded = files.upload()

input_filename = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {input_filename} ({uploaded[input_filename].__len__()} bytes)")

In [ ]:
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
import os

# Set your encryption key
# Option 1: Generate a new random key
ENCRYPTION_KEY = get_random_bytes(32).hex()

# Option 2: Use your existing key (uncomment and paste)
# ENCRYPTION_KEY = "your_64_character_hex_key_here"

print("=" * 60)
print("🔑 YOUR ENCRYPTION KEY (SAVE THIS!)")
print("=" * 60)
print(ENCRYPTION_KEY)
print("=" * 60)
print("\n⚠️ Save this key securely! You'll need it to decrypt your playlist.")
print("Add it as a GitHub Secret: ENCRYPT_KEY")

In [ ]:
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
from google.colab import files

def encrypt_playlist(input_file, output_file, key_hex):
    """Encrypt M3U playlist with AES-256-GCM."""
    key = bytes.fromhex(key_hex)
    if len(key) != 32:
        raise ValueError("Key must be exactly 32 bytes (64 hex characters)")

    with open(input_file, 'rb') as f:
        plaintext = f.read()

    iv = get_random_bytes(12)
    cipher = AES.new(key, AES.MODE_GCM, nonce=iv)
    ciphertext, tag = cipher.encrypt_and_digest(plaintext)

    with open(output_file, 'wb') as f:
        f.write(iv)
        f.write(tag)
        f.write(ciphertext)

    print(f"✅ Encrypted: {input_file} → {output_file}")
    print(f"   Original size: {len(plaintext)} bytes")
    print(f"   Encrypted size: {len(iv) + len(tag) + len(ciphertext)} bytes")

# Encrypt
output_filename = input_filename.replace('.m3u', '.enc').replace('.m3u8', '.enc')
encrypt_playlist(input_filename, output_filename, ENCRYPTION_KEY)

# Download the encrypted file
print("\n📥 Downloading encrypted file...")
files.download(output_filename)

## 📋 Next Steps

1. **Download the `.enc` file** (should have started automatically)
2. **Upload to GitHub**: Push the `.enc` file to `input/` folder in your repo fork
3. **Set GitHub Secret**: Go to repo Settings → Secrets → Actions → New repository secret
   - Name: `ENCRYPT_KEY`
   - Value: Your 64-character hex key from above
4. **The engine will auto-decrypt** and merge your streams on the next run!

---

### 🔓 Decrypting (Verification)

To verify your encrypted file, run the decryption cell below.

In [ ]:
from Crypto.Cipher import AES

def decrypt_file(input_file, output_file, key_hex):
    """Decrypt an AES-256-GCM encrypted file."""
    key = bytes.fromhex(key_hex)

    with open(input_file, 'rb') as f:
        iv = f.read(12)
        tag = f.read(16)
        ciphertext = f.read()

    cipher = AES.new(key, AES.MODE_GCM, nonce=iv)
    plaintext = cipher.decrypt_and_verify(ciphertext, tag)

    with open(output_file, 'wb') as f:
        f.write(plaintext)

    print(f"✅ Decrypted: {input_file} → {output_file}")

# Decrypt to verify
verify_filename = "decrypted_verify.m3u"
decrypt_file(output_filename, verify_filename, ENCRYPTION_KEY)

# Show first few lines
print("\nFirst 10 lines of decrypted file:")
with open(verify_filename) as f:
    for i, line in enumerate(f):
        if i >= 10: break
        print(line.rstrip())